# 도로 침수 라벨 신뢰도 향상 — 센서별 진위 감사(audit)

> 2026-06-19 · 베이스라인 후속(정제 라벨 신뢰도 ↑) · 강우 불필요

**문제** — `flood_t6` 양성의 대부분이 소수 센서에 집중되고, 그 상당수가 stuck/단일값/겨울 아티팩트(베이스라인 §C에서 stuck 거품 ~20%+ 확인).

**방법** — 양성 생성 센서를 **시계열 구조**로 진짜 상습침수 vs 아티팩트로 분류(강우 없이 가능):
- `stuck_pos`(양성이 stuck run에 속한 비율), `max_ep`(최장 연속 양성 길이=1day↑면 비물리적), `top1val`/`nuniq`(단일값 고정 여부), `summer/winter`(장마 계절성), `n_ep`(에피소드 수).
- 진짜 침수 = **솟았다 회복하는 에피소드 + 값 다양 + 여름≥겨울**. 아티팩트 = stuck/장기run/단일값/겨울지배.


In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, matplotlib.font_manager as fm
from pathlib import Path
OUT=Path("dataset/processed/eda_based"); FIG=Path("reports/figures_label"); FIG.mkdir(parents=True,exist_ok=True)
korf=next((c for c in ['NanumGothic','Malgun Gothic','AppleGothic','Noto Sans CJK KR'] if c in {f.name for f in fm.fontManager.ttflist}),None)
if korf: plt.rcParams['font.family']=korf
plt.rcParams['axes.unicode_minus']=False
df=pd.read_parquet(OUT/"road_panel_10min.parquet",
   columns=['sensor_id','ts10','road_max','stuck_frac','month','split','flood_t6','flood_t6_sus','grade']).sort_values(['sensor_id','ts10']).reset_index(drop=True)
df['win']=df.month.isin([12,1,2]); df['sm']=df.month.isin([6,7,8,9])
rows=[]
for sid,s in df.groupby('sensor_id',sort=False):
    npos=int(s.flood_t6.sum())
    if npos==0: continue
    pos=s[s.flood_t6==1]; f=s.flood_t6.values
    chg=np.r_[True,f[1:]!=f[:-1]]; rid=np.cumsum(chg)-1; rl=np.bincount(rid)[rid]
    ep=[rl[i] for i in np.flatnonzero(chg&(f==1))]
    rows.append(dict(sensor_id=sid,grade=s.grade.iloc[0],n_pos=npos,n_sus=int(s.flood_t6_sus.sum()),
        stuck_pos=round((pos.stuck_frac>0).mean(),2),summer=round(pos['sm'].mean(),2),winter=round(pos['win'].mean(),2),
        n_ep=len(ep),max_ep=int(max(ep)) if ep else 0,top1val=round(pos.road_max.value_counts(normalize=True).iloc[0],2),nuniq=int(pos.road_max.nunique())))
A=pd.DataFrame(rows).sort_values('n_pos',ascending=False).reset_index(drop=True)
A['artifact']=(A.stuck_pos>=0.5)|(A.max_ep>=144)|(A.top1val>=0.9)|(A.winter>=0.6)
A['real']=(~A.artifact)&(A.summer>=A.winter)&(A.winter<0.4)&(A.n_ep>=8)&(A.nuniq>=6)
A['판정']=np.where(A.real,'진짜상습',np.where(A.artifact,'아티팩트','애매'))
A.to_parquet(OUT/"road_flood_sensor_audit.parquet",index=False)
tot=A.n_pos.sum()
print("센서 판정:",A['판정'].value_counts().to_dict())
print("판정별 양성수(비율):",{k:f"{v} ({v/tot*100:.0f}%)" for k,v in A.groupby('판정').n_pos.sum().items()})
print("아티팩트 사유: stuck %d / 장기run≥1day %d / 단일값 %d / 겨울지배 %d"%((A.stuck_pos>=0.5).sum(),(A.max_ep>=144).sum(),(A.top1val>=0.9).sum(),(A.winter>=0.6).sum()))
A.head(15)


## 대표 센서 시각화 (진짜상습 vs 아티팩트)

road_max 시계열(파랑) + 침수 양성(빨강). 진짜=에피소드형 회복·값 다양, 아티팩트=사각파/stuck블록/단일값.

In [ ]:
reals=A[A['판정']=='진짜상습'].sort_values('n_pos',ascending=False).sensor_id.head(3).tolist()
arts=[x for x in ['면목동 168-2','봉천동 911-14','충정로2가 130-1'] if x in set(df.sensor_id)]
sel=reals+arts
fig,ax=plt.subplots(len(sel),1,figsize=(13,1.7*len(sel)))
for i,sid in enumerate(sel):
    s=df[df.sensor_id==sid]; r=A[A.sensor_id==sid].iloc[0]
    ax[i].plot(s.ts10,s.road_max,lw=.4,color='steelblue'); p=s[s.flood_t6==1]; ax[i].scatter(p.ts10,p.road_max,s=2,color='crimson')
    ax[i].set_title(f"{sid} [{r['판정']}] n_pos={r.n_pos} stuck={r.stuck_pos} 겨울={r.winter} 값종류={r.nuniq} 최장run={r.max_ep}",fontsize=8)
    ax[i].set_ylabel('road_max',fontsize=7)
plt.tight_layout(); plt.savefig(FIG/"01_sensor_examples.png",dpi=110); plt.show()


## 신뢰 라벨 정의 + 적용

**신뢰 라벨** = `진짜상습` 센서의 `flood_t6_sus`(지속). 아티팩트 센서 양성은 모델 타겟에서 제외, 애매는 보류(강우/추가검토 대상).


In [ ]:
trust=set(A[A['판정']=='진짜상습'].sensor_id); art=set(A[A['판정']=='아티팩트'].sensor_id)
df['trust_label']=np.where(df.sensor_id.isin(trust),df.flood_t6_sus,0)
print("현행 flood_t6 양성:",int(df.flood_t6.sum()))
print("신뢰 양성(진짜상습×flood_t6_sus):",int(df.trust_label.sum()),"(%.1f%%)"%(df.trust_label.sum()/df.flood_t6.sum()*100))
print("split별 신뢰 양성:",df.groupby('split').trust_label.sum().astype(int).to_dict())
# 센서 trust 매핑 저장 (모델링에서 조인용)
A[['sensor_id','판정','n_pos','stuck_pos','summer','winter','nuniq','grade']].to_parquet(OUT/"road_flood_sensor_trust.parquet",index=False)
print("저장:",OUT/"road_flood_sensor_trust.parquet","|",OUT/"road_flood_sensor_audit.parquet")


## 애매 36센서 추가 판별 (강우 없이)

애매 = 아티팩트도 진짜도 확정 못 한(대부분 양성 적은) 센서. 두 신호로 마저 가른다:
- **호우창 동조(lift)**: 진짜상습 7센서가 침수한 시각 ±6h를 '실제 호우창'으로 보고, 애매 센서 침수가 그 창에 몰리는 정도. 실제 호우면 여러 지점이 동시 침수 → lift↑. (검증: 아티팩트 센서 lift 중앙 ≈0)
- **n_storm = 침수건수 × 동조율**: 실제 호우와 겹친 침수 '건수'. 양성 2개에 lift만 높은 건 약하므로 건수로 보정.
- 승격(→진짜상습): n_storm≥5 & 여름 & lift≥3. 강등(→아티팩트): lift≤1.5 또는 겨울지배. 그 외: **미결**(강우 필요).


In [ ]:
trusted=set(A[A['판정']=='진짜상습'].sensor_id); amb=A[A['판정']=='애매'].sensor_id.tolist(); arts=set(A[A['판정']=='아티팩트'].sensor_id)
tf=np.unique(df[df.sensor_id.isin(trusted)&(df.flood_t6_sus==1)].ts10.values.astype('datetime64[m]').astype(np.int64))
def in_storm(ts):
    pos=np.searchsorted(tf,ts); out=np.zeros(len(ts),bool)
    for k in (pos-1,pos):
        kk=np.clip(k,0,len(tf)-1); out|=np.abs(tf[kk]-ts)<=360
    return out
def metr(sid):
    s=df[df.sensor_id==sid]; ts=s.ts10.values.astype('datetime64[m]').astype(np.int64)
    st=in_storm(ts); base=st.mean(); pm=s.flood_t6.values==1
    cooc=st[pm].mean() if pm.sum() else 0.0
    return base,cooc,(cooc/base if base>0 else 0),int(pm.sum()),int(round(pm.sum()*cooc))
rows=[]
for sid in amb:
    b,c,l,n,ns=metr(sid); w=A.loc[A.sensor_id==sid,'winter'].iloc[0]
    rows.append(dict(sensor_id=sid,n_pos=n,n_storm=ns,winter=w,lift=round(l,1),
        판정2=('진짜상습' if (ns>=5 and w<0.4 and l>=3) else '아티팩트' if (l<=1.5 or w>=0.5) else '미결')))
R=pd.DataFrame(rows).sort_values(['판정2','n_storm'],ascending=[True,False])
promo=R[R['판정2']=='진짜상습'].sensor_id.tolist(); demo=R[R['판정2']=='아티팩트'].sensor_id.tolist()
print("애매 재판정:",R['판정2'].value_counts().to_dict())
print("승격→진짜상습:",promo)
print("강등→아티팩트:",len(demo),"개 | 미결:",int((R['판정2']=='미결').sum()),"개(대부분 양성<5·호우동조 1건 → 강우 필요)")

# 최종 판정 반영 + 저장
m2=dict(zip(R.sensor_id,R['판정2']))
A['판정_final']=A.apply(lambda r: m2.get(r.sensor_id,r['판정']),axis=1)
A.to_parquet(OUT/"road_flood_sensor_audit.parquet",index=False)
A[['sensor_id','판정','판정_final','n_pos','stuck_pos','winter','nuniq','grade']].to_parquet(OUT/"road_flood_sensor_trust.parquet",index=False)
new_trusted=set(A[A['판정_final']=='진짜상습'].sensor_id)
df['trust_label']=np.where(df.sensor_id.isin(new_trusted),df.flood_t6_sus,0)
print("\n신뢰 센서: 7 → %d | 신뢰 라벨 양성: 1186 → %d"%(len(new_trusted),int(df.trust_label.sum())))

# 검증 그림 + 승격센서 1분 파형
fig=plt.figure(figsize=(13,6))
ax0=fig.add_subplot(2,1,1)
art_l=[metr(s)[2] for s in arts if df[df.sensor_id==s].flood_t6.sum()>0]
ax0.hist([x for x in art_l if x<25],bins=20,alpha=.6,label='아티팩트',color='gray')
ax0.hist([x for x in R.lift if x<25],bins=20,alpha=.6,label='애매',color='orange')
ax0.axvline(3,color='r',ls='--',lw=1); ax0.set_xlabel('호우창 동조 lift'); ax0.set_title('검증: 아티팩트 lift≈0 vs 애매 (빨강=승격 컷)'); ax0.legend()
if promo:
    rc=pd.read_parquet(OUT/"road_cleaned.parquet",columns=['sensor_id','timestamp','level_clean'],filters=[('sensor_id','in',promo[:3])])
    for i,sid in enumerate(promo[:3]):
        ax=fig.add_subplot(2,3,4+i); s=rc[rc.sensor_id==sid]
        ax.plot(s.timestamp,s.level_clean,lw=.3,color='steelblue'); ax.set_title(f"승격 {sid} (1분)",fontsize=8)
plt.tight_layout(); plt.savefig(FIG/"02_adjudication.png",dpi=110); plt.show()


## 결론

- **양성의 93%가 아티팩트 센서**(stuck/장기run/단일값/겨울지배). 진짜 상습침수 신뢰 센서 7개를 1차 확정.
- **애매 36센서 추가 판별(호우창 동조)**: 아티팩트 lift≈0 검증 위에서 → **승격 3**(대림동 862-5·성산로 494-30·신영동 165, 호우와 ≥5회 동조·여름·1분 파형 확인), **강등 11**(호우 무동조 또는 겨울지배), **미결 22**(양성<5·호우동조 1건뿐 → 강우 필요).
- **최종 신뢰 센서 10개 / 신뢰 라벨 1,220 양성**(현행 23,875의 5.1%). 산출: `road_flood_sensor_trust.parquet`(`판정_final`).

**의의** — 강우 없이도 시계열·공간동조만으로 라벨 신뢰도를 "93% 오염" → "고신뢰 10센서·1,220양성 + 명시적 아티팩트 목록"으로 정리. 단 **미결 22센서·소수 양성 편중은 강우로만 최종 해결** 가능(추가 판별의 실익은 +34양성으로 제한적 — 핵심은 강우).

**다음** — (강우 확보 시) 미결 센서 강우 교차검증·양성 확장 / 신뢰 10센서 상습침수 지점 리포트.
